# NeceCity - train the detector on Google Colab
Run the cells from top to bottom. Before you start: **Runtime > Change runtime type > T4 GPU**.

What this notebook does: prepares the photos, trains YOLOv8n (starting from COCO-pretrained weights), measures the result,
shows its predictions, exports the ONNX file used by the web app, and downloads it.

## 1. Install

In [ ]:
!pip install -q ultralytics==8.3.28 onnx onnxslim onnxruntime

## 2. Upload `NeceCity_Train_Photos.zip` (the photos and their .txt label files)

In [ ]:
from google.colab import files
uploaded = files.upload()          # choose NeceCity_Train_Photos.zip
zip_name = list(uploaded.keys())[0]
print("Uploaded:", zip_name)

## 3. Prepare the dataset (checks the labels and splits train / validation)

In [ ]:
!unzip -q -o "$zip_name" -d photos

In [ ]:
%%writefile prepare_dataset.py
"""Turns a flat folder of photos + label files into the folder layout YOLO needs.

Input folder (what you have):          Output folder (what YOLO needs):
    P1.jpg  P1.txt                         images/train/...   labels/train/...
    G3.jpg  G3.txt                         images/val/...     labels/val/...
    ...

Each .txt file has one line per object:   class_number x_centre y_centre width height
(all five numbers after the class are fractions from 0 to 1 of the photo size).

Run from the ml folder:
    python prepare_dataset.py --source ../NeceCity_Train_Photos --output data/real_dataset

What it does:
  1. checks every label file (class number 0-5, five numbers, values between 0 and 1)
  2. splits the photos into 80% training and 20% validation
     - the split is "stratified": every problem type is represented in the validation set
     - the random seed is fixed (42), so you get the same split every time
  3. prints how many photos and boxes each class has
"""
import argparse
import random
import shutil
from collections import Counter, defaultdict
from pathlib import Path

CLASS_NAMES = ["pothole", "road_damage", "garbage", "waterlogging", "damaged_road_sign", "open_drain"]
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp"}
VALIDATION_SHARE = 0.20
SEED = 42


def read_labels(label_file: Path) -> list[list[float]]:
    """Reads one label file and stops with a clear message if something is wrong."""
    rows = []
    for number, line in enumerate(label_file.read_text().splitlines(), start=1):
        if not line.strip():
            continue
        parts = line.split()
        if len(parts) != 5:
            raise SystemExit(f"{label_file.name} line {number}: expected 5 numbers, found {len(parts)}")
        class_id = int(float(parts[0]))
        values = [float(v) for v in parts[1:]]
        if not 0 <= class_id < len(CLASS_NAMES):
            raise SystemExit(f"{label_file.name} line {number}: class {class_id} is not between 0 and {len(CLASS_NAMES) - 1}")
        if any(v < 0 or v > 1 for v in values):
            raise SystemExit(f"{label_file.name} line {number}: box values must be between 0 and 1")
        rows.append([class_id] + values)
    return rows


def main() -> None:
    parser = argparse.ArgumentParser(description="Split labelled photos into train and val folders.")
    parser.add_argument("--source", required=True, help="folder with the photos and the .txt label files")
    parser.add_argument("--output", default="data/real_dataset", help="folder to create")
    args = parser.parse_args()

    source, output = Path(args.source), Path(args.output)
    photos = sorted(p for p in source.rglob("*") if p.suffix.lower() in IMAGE_EXTENSIONS)
    if not photos:
        raise SystemExit(f"No photos found in {source}")

    # read and check every photo's labels
    labels = {}
    for photo in photos:
        label_file = photo.with_suffix(".txt")
        if not label_file.exists():
            raise SystemExit(f"Missing label file for {photo.name} (expected {label_file.name})")
        rows = read_labels(label_file)
        if not rows:
            raise SystemExit(f"{label_file.name} is empty - every photo needs at least one box")
        labels[photo] = rows

    # group photos by their most common class, then pick validation photos from each group
    groups = defaultdict(list)
    for photo, rows in labels.items():
        most_common_class = Counter(int(r[0]) for r in rows).most_common(1)[0][0]
        groups[most_common_class].append(photo)

    rng = random.Random(SEED)
    validation = set()
    for class_id, members in sorted(groups.items()):
        rng.shuffle(members)
        take = max(1, round(len(members) * VALIDATION_SHARE))
        validation.update(members[:take])

    # write the folders
    if output.exists():
        shutil.rmtree(output)
    for photo in photos:
        part = "val" if photo in validation else "train"
        (output / "images" / part).mkdir(parents=True, exist_ok=True)
        (output / "labels" / part).mkdir(parents=True, exist_ok=True)
        shutil.copy2(photo, output / "images" / part / photo.name)
        shutil.copy2(photo.with_suffix(".txt"), output / "labels" / part / (photo.stem + ".txt"))

    # summary
    print(f"Photos: {len(photos)}  (train {len(photos) - len(validation)}, val {len(validation)})")
    print(f"{'class':20s} {'photos':>7s} {'boxes':>7s} {'boxes in val':>13s}")
    for class_id, name in enumerate(CLASS_NAMES):
        photo_count = sum(any(int(r[0]) == class_id for r in rows) for rows in labels.values())
        box_count = sum(int(r[0]) == class_id for rows in labels.values() for r in rows)
        val_boxes = sum(int(r[0]) == class_id for p in validation for r in labels[p])
        print(f"{name:20s} {photo_count:7d} {box_count:7d} {val_boxes:13d}")
    print(f"Done. Dataset written to {output}")


if __name__ == "__main__":
    main()

In [ ]:
!python prepare_dataset.py --source photos --output data/real_dataset

In [ ]:
yaml_text = '''path: /content/data/real_dataset
train: images/train
val: images/val
names:
  0: pothole
  1: road_damage
  2: garbage
  3: waterlogging
  4: damaged_road_sign
  5: open_drain
'''
open('dataset.yaml', 'w').write(yaml_text)
print(yaml_text)

## 4. Train
This takes a few minutes on the free GPU. Watch the mAP50 value grow.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")         # COCO-pretrained starting point (transfer learning)
model.train(
    data="dataset.yaml", epochs=150, imgsz=640, batch=8, patience=40, seed=42,
    degrees=10, fliplr=0.5, hsv_s=0.7, hsv_v=0.4, scale=0.5, mosaic=1.0,
    project="runs", name="nececity_yolo",
)
best_path = str(model.trainer.best)
print("Best model:", best_path)

## 5. Measure the result on the validation photos

In [ ]:
best = YOLO(best_path)
results = best.val(data="dataset.yaml", imgsz=640)
print("Precision    :", round(float(results.box.mp), 3))
print("Recall       :", round(float(results.box.mr), 3))
print("mAP@0.5      :", round(float(results.box.map50), 3))
print("mAP@0.5-0.95 :", round(float(results.box.map), 3))
print()
for class_id, score in zip(results.box.ap_class_index, results.box.ap50):
    print(f"{best.names[int(class_id)]:20s} mAP@0.5 = {float(score):.3f}")

## 6. Look at the predictions on the validation photos

In [ ]:
from PIL import Image
from IPython.display import display

for r in best.predict(source="data/real_dataset/images/val", conf=0.25, verbose=False):
    display(Image.fromarray(r.plot()[..., ::-1]))

## 7. Export to ONNX (the file the web app uses) and check it

In [ ]:
import numpy as np, onnxruntime as ort

onnx_path = best.export(format="onnx", imgsz=640, opset=12)
session = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
output = session.run(None, {session.get_inputs()[0].name: np.zeros((1, 3, 640, 640), np.float32)})[0]
print("ONNX output shape:", output.shape, "(expected (1, 10, 8400))")

## 8. Download the two files
Put both in the `ml/weights` folder of your project.

In [ ]:
import shutil
shutil.copy(best_path, "nececity_yolo.pt")
shutil.copy(onnx_path, "nececity_yolo.onnx")
files.download("nececity_yolo.onnx")
files.download("nececity_yolo.pt")